# AI for Data Cleaning & Classification — Groq + Llama on Databricks
**Session Duration:** 60 minutes | **Stack:** Python · Groq API · Llama 3.1 · Databricks
**Author:** ANALYTICSWITHANAND | Anand Kumar Jha

| Segment | Topic | Time |
|---------|-------|------|
| 1 | Setup & Motivation | 0–10 min |
| 2 | Data Cleaning with LLM | 10–25 min |
| 3 | Zero-Shot Classification | 25–40 min |
| 4 | Evaluation & Cost Awareness | 40–52 min |
| 5 | Production Patterns & Wrap-up | 52–60 min |

---
## Segment 1 — Setup & Motivation (0–10 min)
**Why Groq?** Free tier · 30 RPM · 500K tokens/day · No credit card · Fastest LLM inference

Get your free key at: **console.groq.com** → Sign up → API Keys → Create API key

In [0]:
# Install Groq SDK
%pip install groq -q
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from groq import Groq
import pandas as pd
import json
import re
import time
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Paste your Groq key from console.groq.com
# GROQ_API_KEY = ""  #unccomment and paste your key here

client = Groq(api_key=GROQ_API_KEY)

def call_llm(prompt: str, max_tokens: int = 256) -> str:
    """
    Single wrapper function — replaces model.generate_content() everywhere.
    temperature=0 for deterministic, consistent output.
    """
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()

# Smoke test
print(call_llm("Reply with exactly: LLM ready on Databricks"))

LLM ready on Databricks.


databricks secrets create-scope my-secret-scope
databricks secrets put-secret my-secret-scope db-password

to access

db_pswd = dbutils.secret.get(scope='my-secret-scope.,key)

---
## Segment 2 — Data Cleaning with LLM (10–25 min)
**Goal:** fix messy company names and countries using LLM prompts

**Key rule:** always ask the model to return ONLY the cleaned value — no explanation, no quotes.

In [0]:
raw_data = {
    "id": list(range(1, 21)),
    "company_name": [
        "apple inc.", "APPLE INC", "Apple Incorporated", "appl inc",
        "google llc", "Gooogle LLC", "GOOGLE", "Google L.L.C.",
        "microsoft corp", "Microsoft Corporation", "MSFT Corp", "microsoft",
        "amazon.com inc", "AMAZON INC.", "Amazon", "amzn inc",
        "meta platforms", "META PLATFORMS INC", "Facebook/Meta", "meta inc.",
    ],
    "country": [
        "usa", "U.S.A", "United States", "US",
        "usa", "united states of america", "US", "U.S.",
        "usa", "United States", "US", "u.s.a.",
        "usa", "US", "United States", "U.S.A.",
        "usa", "US", "United States", "u.s.",
    ],
    "support_ticket": [
        "My bill is wrong and I was overcharged by $50!!",
        "cant login to my account, password reset not working",
        "order #12345 hasnt arrived yet its been 2 weeks",
        "need refund for damaged product",
        "WEBSITE IS DOWN PLEASE HELP URGENT",
        "how do i cancel my subscription?",
        "i was charged twice for the same order",
        "app keeps crashing on iphone",
        "where is my package? tracking shows delivered but nothing here",
        "want to upgrade my plan to enterprise",
        "got wrong item in my shipment",
        "billing statement is confusing please explain",
        "forgot password and email also changed",
        "product quality is very poor not as described",
        "need invoice for my purchase last month",
        "technical error code 500 on checkout page",
        "delivery was damaged and box was open",
        "how to export my data from the platform",
        "promo code not working at checkout",
        "account locked after too many login attempts",
    ],
    "revenue_usd": [
        "1,200.50", "$800", "950.0", "1100",
        "2,400.00", "$1,800.75", "2100", "2,050",
        "3,500.25", "$3,200", "3100.5", "3,450.00",
        "4,800.00", "$4,500.50", "4,200", "4,750",
        "5,100.25", "$5,000", "4,900.75", "5,050",
    ],
}
df = pd.DataFrame(raw_data)
print(f"Dataset shape: {df.shape}")
df.head(10)

Dataset shape: (20, 5)


,id,company_name,country,support_ticket,revenue_usd
0,1,apple inc.,usa,My bill is wrong and I was overcharged by $50!!,"1,200.50"
1,2,APPLE INC,U.S.A,"cant login to my account, password reset not w...",$800
2,3,Apple Incorporated,United States,order #12345 hasnt arrived yet its been 2 weeks,950.0
3,4,appl inc,US,need refund for damaged product,1100
4,5,google llc,usa,WEBSITE IS DOWN PLEASE HELP URGENT,"2,400.00"
5,6,Gooogle LLC,united states of america,how do i cancel my subscription?,"$1,800.75"
6,7,GOOGLE,US,i was charged twice for the same order,2100
7,8,Google L.L.C.,U.S.,app keeps crashing on iphone,"2,050"
8,9,microsoft corp,usa,where is my package? tracking shows delivered ...,"3,500.25"
9,10,Microsoft Corporation,United States,want to upgrade my plan to enterprise,"$3,200"


In [0]:
# Teaching moment: vague vs precise prompt
print("=== VAGUE PROMPT ===")
print("  appl inc ->", call_llm("Fix this company name: appl inc"))

print("\n=== PRECISE PROMPT ===")
precise = call_llm("""You are a data cleaning assistant.
Clean this company name. Return ONLY the cleaned value — no explanation, no quotes.
Examples:
  apple inc.  ->  Apple
  GOOGLE LLC  ->  Google
Input: appl inc
Cleaned:""")
print("  appl inc ->", precise)
print("\nLesson: precise prompt = consistent, parseable output")

=== VAGUE PROMPT ===
  appl inc -> Here are a few suggestions to improve the company name "appl inc":

1. **Applify Inc.**: This name adds a suffix "-ify" which implies transformation or improvement, making it more appealing.
2. **Applix Inc.**: This name adds a unique twist to the original name, making it more memorable.
3. **Applio Inc.**: This name adds a Latin-inspired suffix "-io" which gives it a more sophisticated and modern feel.
4. **Applium Inc.**: This name adds a unique suffix "-ium" which gives it a more scientific and innovative feel.
5. **Applify Solutions Inc.**: This name adds a descriptive phrase "Solutions" which gives it a more clear and concise meaning.

Choose the one that best fits your company's brand and values.

=== PRECISE PROMPT ===
  appl inc -> Apple

Lesson: precise prompt = consistent, parseable output


In [0]:
def clean_with_llm(value: str, field_type: str, examples: str = "") -> str:
    """Clean a single field value using Groq LLM."""
    if pd.isna(value) or str(value).strip() == "":
        return value
    prompt = f"""You are a precise data cleaning assistant.
Task: Clean and standardise this {field_type}.
Rules:
- Return ONLY the cleaned value — no explanation, no quotes, no punctuation at end
- Correct spelling errors and inconsistent casing
- Use the most official/standard form
- Do not add information not present in the input
{f'Examples:{examples}' if examples else ''}
Input: {value}
Cleaned:"""
    try:
        return call_llm(prompt, max_tokens=64)
    except Exception as e:
        print(f"  Error on '{value}': {e}")
        return value

print("clean_with_llm() defined")

clean_with_llm() defined


In [0]:
company_examples = """
  apple inc.      ->  Apple
  GOOGLE          ->  Google
  microsoft corp  ->  Microsoft
  amazon.com inc  ->  Amazon"""

print("Cleaning company_name...")
df["company_name_clean"] = df["company_name"].apply(
    lambda x: clean_with_llm(x, "Fortune 500 company name", company_examples)
)
time.sleep(1)

country_examples = """
  usa    ->  United States
  U.S.A  ->  United States
  u.s.   ->  United States"""

print("Cleaning country...")
df["country_clean"] = df["country"].apply(
    lambda x: clean_with_llm(x, "country name (full official English name)", country_examples)
)

# Revenue — rule-based (no LLM needed for structured formats)
df["revenue_clean"] = (
    df["revenue_usd"].str.replace(r"[$,]", "", regex=True).str.strip().astype(float)
)

print("\nBefore -> After:")
df[["company_name", "company_name_clean", "country", "country_clean", "revenue_usd", "revenue_clean"]]

Cleaning company_name...
Cleaning country...

Before -> After:


,company_name,company_name_clean,country,country_clean,revenue_usd,revenue_clean
0,apple inc.,Apple,usa,United States,"1,200.50",1200.50
1,APPLE INC,Apple,U.S.A,United States,$800,800.00
2,Apple Incorporated,Apple,United States,United States,950.0,950.00
3,appl inc,Apple,US,United States,1100,1100.00
4,google llc,Google,usa,United States,"2,400.00",2400.00
5,Gooogle LLC,Google,united states of america,United States of America,"$1,800.75",1800.75
6,GOOGLE,Google,US,United States,2100,2100.00
7,Google L.L.C.,Google LLC,U.S.,United States,"2,050",2050.00
8,microsoft corp,Microsoft,usa,United States,"3,500.25",3500.25
9,Microsoft Corporation,Microsoft Corporation,United States,United States,"$3,200",3200.00


---
## Segment 3 — Zero-Shot Text Classification (25–40 min)
**Zero-shot** = no labelled training data needed — the model already understands business language.

**Golden rules:**
- Always say `Return ONLY valid JSON` in the prompt
- Always wrap `json.loads()` in try/except
- Strip markdown fences the model sometimes adds despite instructions

In [0]:
import pandas as pd
import json
import re

TICKET_CATEGORIES = [
    "Billing",
    "Technical",
    "Shipping",
    "Returns",
    "Account",
    "Other"
]

def classify_ticket(text: str) -> dict:
    """
    Single-label ticket classification.
    Returns: {category, confidence, reason}
    """

    # Handle empty input
    if pd.isna(text) or str(text).strip() == "":
        return {
            "category": "Other",
            "confidence": "low",
            "reason": "Empty input"
        }

    prompt = f"""
You are a customer support ticket classifier.

Classify the following customer ticket into exactly ONE category.

Allowed categories:
{', '.join(TICKET_CATEGORIES)}

Return ONLY a valid JSON object in exactly this format:

{{
    "category": "Billing",
    "confidence": "high",
    "reason": "Customer reports an incorrect billing charge."
}}

Rules:
- category must be exactly one of: {', '.join(TICKET_CATEGORIES)}
- confidence must be exactly one of: high, medium, low
- reason must be one sentence under 15 words
- Do not return markdown
- Do not return code fences
- Do not return any text before or after the JSON

Ticket:
{text}

JSON output:
"""

    try:
        # Call LLM
        raw = call_llm(prompt, max_tokens=128)

        # Debug: See exactly what the LLM returned
        print("  Raw LLM output:", repr(raw))

        # Remove markdown code fences if LLM ignored instructions
        cleaned = re.sub(
            r"```(?:json)?|```",
            "",
            raw,
            flags=re.IGNORECASE
        ).strip()

        # Parse JSON
        result = json.loads(cleaned)

        # Validate required fields
        if "category" not in result:
            raise ValueError("Missing category")

        if "confidence" not in result:
            raise ValueError("Missing confidence")

        if "reason" not in result:
            raise ValueError("Missing reason")

        # Validate category
        if result["category"] not in TICKET_CATEGORIES:
            raise ValueError(
                f"Invalid category: {result['category']}"
            )

        # Validate confidence
        if result["confidence"] not in [
            "high",
            "medium",
            "low"
        ]:
            raise ValueError(
                f"Invalid confidence: {result['confidence']}"
            )

        return result

    except json.JSONDecodeError as e:
        print(f"  JSON parse error: {e}")
        print(f"  Raw response: {repr(raw)}")

        return {
            "category": "Other",
            "confidence": "low",
            "reason": "Invalid JSON response"
        }

    except ValueError as e:
        print(f"  Validation error: {e}")

        return {
            "category": "Other",
            "confidence": "low",
            "reason": "Validation failed"
        }

    except Exception as e:
        print(f"  API error: {e}")

        return {
            "category": "Other",
            "confidence": "low",
            "reason": "API error"
        }


# Test tickets
test_tickets = [
    "My bill is wrong and I was overcharged by $50!!",
    "WEBSITE IS DOWN PLEASE HELP URGENT",
    "order #12345 hasnt arrived yet its been 2 weeks",
]

print("Single-label demo:\n")

for ticket in test_tickets:
    print(f"Ticket: {ticket}")
    result = classify_ticket(ticket)
    print(f"Result: {result}\n")

Single-label demo:

Ticket: My bill is wrong and I was overcharged by $50!!
  Raw LLM output: '{\n    "category": "Billing",\n    "confidence": "high",\n    "reason": "Customer reports an incorrect billing charge."\n}'
Result: {'category': 'Billing', 'confidence': 'high', 'reason': 'Customer reports an incorrect billing charge.'}

Ticket: WEBSITE IS DOWN PLEASE HELP URGENT
  Raw LLM output: '{\n    "category": "Technical",\n    "confidence": "high",\n    "reason": "Customer reports website is down and requires urgent assistance."\n}'
Result: {'category': 'Technical', 'confidence': 'high', 'reason': 'Customer reports website is down and requires urgent assistance.'}

Ticket: order #12345 hasnt arrived yet its been 2 weeks
  Raw LLM output: '{\n    "category": "Shipping",\n    "confidence": "high",\n    "reason": "Customer reports delayed shipping of order #12345."\n}'
Result: {'category': 'Shipping', 'confidence': 'high', 'reason': 'Customer reports delayed shipping of order #12345.'}



In [0]:
TICKET_CATEGORIES = ["Billing", "Technical", "Shipping", "Returns", "Account", "Other"]

def classify_ticket(text: str) -> dict:
    """Single-label ticket classification. Returns {category, confidence, reason}."""
    if pd.isna(text) or str(text).strip() == "":
        return {"category": "Other", "confidence": "low", "reason": "Empty input"}

    prompt = f"""You are a customer support ticket classifier.
Classify the ticket below into exactly ONE of: {', '.join(TICKET_CATEGORIES)}
Rules:
- Return ONLY valid JSON — no markdown, no code fences, no extra text
- confidence must be exactly: high, medium, or low
- reason must be one sentence under 15 words
Ticket: {text}
JSON output:"""

    try:
        raw = call_llm(prompt, max_tokens=128)
        raw = re.sub(r"```(?:json)?|```", "", raw).strip()
        result = json.loads(raw)
        assert "category" in result and "confidence" in result
        return result
    except (json.JSONDecodeError, AssertionError) as e:
        print(f"  Parse error: {e}")
        return {"category": "Other", "confidence": "low", "reason": "Parse failed"}
    except Exception as e:
        print(f"  API error: {e}")
        return {"category": "Other", "confidence": "low", "reason": "API error"}


# Test on 3 tickets
test_tickets = [
    "My bill is wrong and I was overcharged by $50!!",
    "WEBSITE IS DOWN PLEASE HELP URGENT",
    "order #12345 hasnt arrived yet its been 2 weeks",
]
print("Single-label demo:\n")
for t in test_tickets:
    print(f"  Ticket: {t}")
    print(f"  Result: {classify_ticket(t)}\n")

Single-label demo:

  Ticket: My bill is wrong and I was overcharged by $50!!
  Parse error: 
  Result: {'category': 'Other', 'confidence': 'low', 'reason': 'Parse failed'}

  Ticket: WEBSITE IS DOWN PLEASE HELP URGENT
  Result: {'category': 'Technical', 'confidence': 'high', 'reason': 'Customer reports website is down and requires immediate assistance.'}

  Ticket: order #12345 hasnt arrived yet its been 2 weeks
  Parse error: 
  Result: {'category': 'Other', 'confidence': 'low', 'reason': 'Parse failed'}



In [0]:
def classify_ticket_multilabel(text: str) -> dict:
    """Multi-label: ticket can belong to more than one category."""
    if pd.isna(text) or str(text).strip() == "":
        return {"categories": ["Other"], "primary": "Other", "confidence": "low"}

    prompt = f"""You are a customer support ticket classifier.
Classify the ticket. It may belong to ONE OR MORE of: {', '.join(TICKET_CATEGORIES)}
Rules:
- Return ONLY valid JSON, no markdown fences
- categories: array of ALL matching categories (at least one)
- primary: the single most important category
- confidence: high | medium | low
Ticket: {text}
JSON output:"""

    try:
        raw = call_llm(prompt, max_tokens=128)
        raw = re.sub(r"```(?:json)?|```", "", raw).strip()
        return json.loads(raw)
    except Exception:
        return {"categories": ["Other"], "primary": "Other", "confidence": "low"}


multi_ticket = "i was charged twice for the same order and now my account is locked"
result_ml = classify_ticket_multilabel(multi_ticket)
print(f"Ticket : {multi_ticket}")
print(f"Result : {json.dumps(result_ml, indent=2)}")

Ticket : i was charged twice for the same order and now my account is locked
Result : {
  "categories": [
    "Billing",
    "Account"
  ],
  "primary": "Billing",
  "confidence": "high"
}


In [0]:
print("Classifying all 20 tickets...\n")
classifications = []
for i, row in df.iterrows():
    result = classify_ticket(row["support_ticket"])
    classifications.append(result)
    if (i + 1) % 10 == 0:
        time.sleep(2)
        print(f"  Processed {i+1}/{len(df)}...")

df["ticket_category"]   = [c.get("category",   "Other") for c in classifications]
df["ticket_confidence"] = [c.get("confidence", "low")   for c in classifications]
df["ticket_reason"]     = [c.get("reason",     "")       for c in classifications]

print("\nCategory distribution:")
print(df["ticket_category"].value_counts().to_string())
print("\nResults:")
df[["support_ticket", "ticket_category", "ticket_confidence"]]

Classifying all 20 tickets...

  Raw LLM output: '{\n    "category": "Billing",\n    "confidence": "high",\n    "reason": "Customer reports an incorrect billing charge."\n}'
  Raw LLM output: '{\n    "category": "Account",\n    "confidence": "high",\n    "reason": "Customer reports issues with account login and password reset."\n}'
  Raw LLM output: '{\n    "category": "Shipping",\n    "confidence": "high",\n    "reason": "Customer reports delayed shipping of order #12345."\n}'
  Raw LLM output: '{\n    "category": "Returns",\n    "confidence": "high",\n    "reason": "Customer requests a refund for a damaged product."\n}'
  Raw LLM output: '{\n    "category": "Technical",\n    "confidence": "high",\n    "reason": "Customer reports website is down and requires urgent assistance."\n}'
  Raw LLM output: '{\n    "category": "Account",\n    "confidence": "high",\n    "reason": "Customer requests to cancel their subscription."\n}'
  Raw LLM output: '{\n    "category": "Billing",\n    "confid

,support_ticket,ticket_category,ticket_confidence
0,My bill is wrong and I was overcharged by $50!!,Billing,high
1,"cant login to my account, password reset not w...",Account,high
2,order #12345 hasnt arrived yet its been 2 weeks,Shipping,high
3,need refund for damaged product,Returns,high
4,WEBSITE IS DOWN PLEASE HELP URGENT,Technical,high
5,how do i cancel my subscription?,Account,high
6,i was charged twice for the same order,Billing,high
7,app keeps crashing on iphone,Technical,high
8,where is my package? tracking shows delivered ...,Shipping,high
9,want to upgrade my plan to enterprise,Account,high


---
## Segment 4 — Evaluation & Cost Awareness (40–52 min)
Measure quality with a gold-standard spot check · Track token usage · Build a dedup cache

In [0]:
gold_standard = {
    0: "Billing", 1: "Account", 2: "Shipping", 3: "Returns",
    4: "Technical", 5: "Account", 6: "Billing", 7: "Technical",
}
correct = 0
print(f"{'#':<4} {'Ticket':<45} {'Human':<12} {'LLM':<12} Match")
print("-" * 80)
for idx, human_label in gold_standard.items():
    ticket    = df.loc[idx, "support_ticket"][:42] + "..."
    llm_label = df.loc[idx, "ticket_category"]
    match     = "PASS" if llm_label == human_label else "FAIL"
    if llm_label == human_label: correct += 1
    print(f"{idx:<4} {ticket:<45} {human_label:<12} {llm_label:<12} {match}")
print(f"\nAccuracy: {correct}/{len(gold_standard)} = {correct/len(gold_standard)*100:.1f}%")

#    Ticket                                        Human        LLM          Match
--------------------------------------------------------------------------------
0    My bill is wrong and I was overcharged by ... Billing      Billing      PASS
1    cant login to my account, password reset n... Account      Account      PASS
2    order #12345 hasnt arrived yet its been 2 ... Shipping     Shipping     PASS
3    need refund for damaged product...            Returns      Returns      PASS
4    WEBSITE IS DOWN PLEASE HELP URGENT...         Technical    Technical    PASS
5    how do i cancel my subscription?...           Account      Account      PASS
6    i was charged twice for the same order...     Billing      Billing      PASS
7    app keeps crashing on iphone...               Technical    Technical    PASS

Accuracy: 8/8 = 100.0%


In [0]:
def classify_with_usage(text: str) -> tuple:
    """Returns (result, usage_info) — Groq free tier: $0. Paid: ~$0.05/1M tokens."""
    prompt = f"""Classify into one of: {', '.join(TICKET_CATEGORIES)}
Return ONLY valid JSON: {{"category": "...", "confidence": "high|medium|low"}}
Ticket: {text}
JSON:"""
    resp  = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0, max_tokens=64,
    )
    usage = resp.usage
    usage_info = {
        "prompt_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens":  usage.total_tokens,
        "cost_usd":      round(usage.total_tokens * 0.05 / 1_000_000, 8),
    }
    try:
        raw    = re.sub(r"```(?:json)?|```", "", resp.choices[0].message.content.strip())
        result = json.loads(raw.strip())
    except Exception:
        result = {"category": "Other", "confidence": "low"}
    return result, usage_info

print("Token usage tracking demo:\n")
total_cost = 0.0
for ticket in df["support_ticket"].iloc[:3]:
    result, usage = classify_with_usage(ticket)
    total_cost += usage["cost_usd"]
    print(f"  Ticket  : {ticket[:55]}...")
    print(f"  Category: {result.get('category')} | Tokens: {usage['total_tokens']} | Cost: ${usage['cost_usd']:.8f}")
    print()
print(f"3 tickets: ${total_cost:.8f}")
print(f"1M rows  : ${total_cost / 3 * 1_000_000:.2f}  (Groq free tier = $0)")

Token usage tracking demo:

  Ticket  : My bill is wrong and I was overcharged by $50!!...
  Category: Billing | Tokens: 103 | Cost: $0.00000515

  Ticket  : cant login to my account, password reset not working...
  Category: Account | Tokens: 101 | Cost: $0.00000505

  Ticket  : order #12345 hasnt arrived yet its been 2 weeks...
  Category: Returns | Tokens: 104 | Cost: $0.00000520

3 tickets: $0.00001540
1M rows  : $5.13  (Groq free tier = $0)


In [0]:
class LLMCache:
    """Dedup cache — skip API call for repeated values. Saves 60-80% of calls."""
    def __init__(self):
        self._cache = {}
        self.hits = self.misses = 0

    def get_or_call(self, value, field_type, prompt_fn):
        key = (str(value).strip().lower(), field_type)
        if key in self._cache:
            self.hits += 1
            return self._cache[key]
        self.misses += 1
        result = prompt_fn(value)
        self._cache[key] = result
        return result

    def stats(self):
        total = self.hits + self.misses
        rate  = self.hits / total * 100 if total else 0
        print(f"hits: {self.hits} | misses: {self.misses} | hit rate: {rate:.1f}% | API calls saved: {self.hits}")


# Demo: 20 rows but only 5 unique company names
cache = LLMCache()
for company in df["company_name"]:
    cache.get_or_call(company, "company_name", lambda v: clean_with_llm(v, "company name"))
cache.stats()

hits: 0 | misses: 20 | hit rate: 0.0% | API calls saved: 0


---
## Segment 5 — Production Patterns & Wrap-up (52–60 min)
Scale with Pandas UDF · Write to Delta Lake · Architecture review

In [0]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType

GROQ_KEY_BC = spark.sparkContext.broadcast(GROQ_API_KEY)

@pandas_udf(StringType())
def classify_tickets_udf(ticket_series: pd.Series) -> pd.Series:
    """Pandas UDF: runs on each Spark partition. Use for datasets > 10K rows."""
    from groq import Groq
    worker_client = Groq(api_key=GROQ_KEY_BC.value)
    results = []
    for text in ticket_series:
        if pd.isna(text):
            results.append("Other")
            continue
        try:
            prompt = f"Classify into [Billing,Technical,Shipping,Returns,Account,Other]. Return ONLY the category word.\nTicket: {text}\nCategory:"
            resp   = worker_client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": prompt}],
                temperature=0, max_tokens=16,
            )
            results.append(resp.choices[0].message.content.strip())
        except Exception:
            results.append("Other")
    return pd.Series(results)


spark_df = spark.createDataFrame(df[["id", "support_ticket"]])
result_df = spark_df.withColumn("ticket_category_udf", classify_tickets_udf(F.col("support_ticket")))
result_df.select("id", "support_ticket", "ticket_category_udf").show(5, truncate=50)

---------------------------------------------------------------------------
PySparkAttributeError                     Traceback (most recent call last)
File <command-6844163800043143>, line 4
      1 from pyspark.sql.functions import pandas_udf
      2 from pyspark.sql.types import StringType
----> 4 GROQ_KEY_BC = spark.sparkContext.broadcast(GROQ_API_KEY)
      6 @pandas_udf(StringType())
      7 def classify_tickets_udf(ticket_series: pd.Series) -> pd.Series:
      8     """Pandas UDF: runs on each Spark partition. Use for datasets > 10K rows."""

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:1130, in SparkSession.__getattr__(self, name)
   1128 def __getattr__(self, name: str) -> Any:
   1129     if name in ["_jsc", "_jconf", "_jvm", "_jsparkSession", "sparkContext", "newSession"]:
-> 1130         raise PySparkAttributeError(
   1131             errorClass="JVM_ATTRIBUTE_NOT_SUPPORTED", messageParameters={"attr_name": name}
   1132         )
   

In [0]:
from datetime import datetime

final_df = df[[
    "id", "company_name", "company_name_clean",
    "country", "country_clean",
    "support_ticket", "ticket_category", "ticket_confidence",
    "revenue_usd", "revenue_clean",
]].copy()
final_df["processed_at"]     = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
final_df["model_used"]       = "groq/llama-3.1-8b-instant"
final_df["pipeline_version"] = "1.0.0"

# Community Edition: use simple table name (no catalog prefix)
(
    spark.createDataFrame(final_df)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cleaned_tickets_groq")
)

print("Delta table written: cleaned_tickets_groq")
spark.table("cleaned_tickets_groq").show(3, truncate=40)

/home/spark-a1b53c82-62d3-4079-b1a6-cd/.ipykernel/307/command-6844163800043144-551464652:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  final_df["processed_at"]     = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


Delta table written: cleaned_tickets_groq
+---+------------------+------------------+-------------+-------------+----------------------------------------+---------------+-----------------+-----------+-------------+-------------------+-------------------------+----------------+
| id|      company_name|company_name_clean|      country|country_clean|                          support_ticket|ticket_category|ticket_confidence|revenue_usd|revenue_clean|       processed_at|               model_used|pipeline_version|
+---+------------------+------------------+-------------+-------------+----------------------------------------+---------------+-----------------+-----------+-------------+-------------------+-------------------------+----------------+
|  1|        apple inc.|             Apple|          usa|United States|My bill is wrong and I was overcharge...|          Other|              low|   1,200.50|       1200.5|2026-07-13 18:17:01|groq/llama-3.1-8b-instant|           1.0.0|
|  2|         

In [0]:
summary = """
╔══════════════════════════════════════════════════════════╗
║  SESSION SUMMARY — Groq + Llama on Databricks           ║
╠══════════════════════════════════════════════════════════╣
║  1. SETUP      Groq free key · call_llm() wrapper        ║
║  2. CLEANING   Precise prompt · few-shot examples        ║
║  3. CLASSIFY   Zero-shot · JSON output · try/except      ║
║  4. EVALUATE   Gold-standard · token cost · dedup cache  ║
║  5. PRODUCTION Pandas UDF · Delta Lake write             ║
╠══════════════════════════════════════════════════════════╣
║  Groq free limits: 30 RPM · 500K tokens/day             ║
║  Swap model: llama-3.3-70b-versatile for harder tasks   ║
╚══════════════════════════════════════════════════════════╝
"""
print(summary)


╔══════════════════════════════════════════════════════════╗
║  SESSION SUMMARY — Groq + Llama on Databricks           ║
╠══════════════════════════════════════════════════════════╣
║  1. SETUP      Groq free key · call_llm() wrapper        ║
║  2. CLEANING   Precise prompt · few-shot examples        ║
║  3. CLASSIFY   Zero-shot · JSON output · try/except      ║
║  4. EVALUATE   Gold-standard · token cost · dedup cache  ║
║  5. PRODUCTION Pandas UDF · Delta Lake write             ║
╠══════════════════════════════════════════════════════════╣
║  Groq free limits: 30 RPM · 500K tokens/day             ║
║  Swap model: llama-3.3-70b-versatile for harder tasks   ║
╚══════════════════════════════════════════════════════════╝

